In [2]:
import sys
sys.path.append(rf"/Users/baia/Desktop/PYTHON/mba_dsa_usp_esalq")

from TCC.utils.constantes import *
import matplotlib.pyplot as plt

## BTC RVI (Relative Volatility Index)
- Indicador técnico que mede a direção da volatilidade calculando o desvio padrão dos preços altos e baixos em uma escala de 0 a 100. Valores acima de 50 indicam volatilidade expansiva (convicção de tendência).

- Fonte: https://br.tradingview.com/support/solutions/43000594684/

### HIPÓTESES
- Era Varejo (Período 1): Análise: O RVI mede a direção da volatilidade. Se o RVI > 50, a volatilidade é de alta. O investidor de varejo opera muito por momentum e "FOMO" (medo de ficar de fora). Portanto, quando a volatilidade técnica subia, o varejo entrava em massa, tornando o indicador altamente preditivo.

- Na Era Institucional (Período 2): Instituições usam modelos complexos e não apenas osciladores simples de High/Low. Além disso, a volatilidade do Bitcoin passou a responder a choques de liquidez global (Juros/Dólar). Logo, um oscilador técnico isolado perde força explicativa frente a uma mudança na taxa de juros do FED.

### TRATAMENTO
- Como ele oscila entre 0 e 100 (sua média é ~52), ele já possui limites definidos, o que evita a necessidade de Log. No entanto, para modelos de séries temporais (como VAR), o valor absoluto (ex: 55) importa menos do que a mudança de estado (se a volatilidade está acelerando ou desacelerando). Por isso, a Primeira Diferença é o tratamento ideal.

In [3]:
df_btc_price = pd.read_csv(rf"raw/price_btc.csv")
df_btc_price['Data_UTC'] = pd.to_datetime(df_btc_price['time'], unit='s', utc=True,).dt.strftime("%Y-%m-%d")

In [4]:
df_btc_rvi =(
    df_periodo
        .merge(df_btc_price, how='left', on='Data_UTC')
        .assign(RVI_Diff = lambda df: df['RVI'].diff())
        .query("Data_UTC > '2016-12-31'")
        [['Data_UTC','RVI_Diff']]

)
df_btc_rvi


,Data_UTC,RVI_Diff
1,2017-01-01,4.586413
2,2017-01-02,4.467393
3,2017-01-03,3.898178
4,2017-01-04,4.286328
5,2017-01-05,-13.132464
...,...,...
3131,2025-07-28,-4.828541
3132,2025-07-29,-4.671673
3133,2025-07-30,-4.259905
3134,2025-07-31,-5.448235


## HV (Volatilidade)
- A Volatilidade Histórica (HV) é uma medida de dispersão estatística (geralmente o desvio padrão anualizado dos log-retornos).
- Representa a incerteza e o risco de curto prazo do ativo. Investidores institucionais utilizam métricas de volatilidade para ajustar posições (Risk Management), enquanto o varejo muitas vezes é atraído por ela ("Hype").

- Fonte: https://br.tradingview.com/support/solutions/43000589145/

### HIPÓTESES
- Na Era Varejo (Período 1), espera-se que o impacto do HV na dominância seja baixo ou positivo (varejo buscando retornos rápidos).

- Na Era Institucional (Período 2), espera-se que o HV tenha alta relevância no SHAP value, indicando que o mercado agora reage à volatilidade ajustando a exposição ("Flight to Quality").

### TRATAMENTO
- Devido ao comportamento típico de "clusters" de volatilidade com picos extremos (Max: 327 vs Média: 59)
- Para modelagem de séries temporais financeiras com esse perfil "explosivo", o tratamento ideal é o Log-Retorno da Volatilidade.

In [5]:
df_btc_hv =(
    df_periodo
        .merge(df_btc_price, how='left', on='Data_UTC')
        .assign(HV_Log_Ret_btc = lambda df: np.log(df['HV']/df['HV'].shift(1)))
        .query("Data_UTC > '2016-12-31'")
        [['Data_UTC','HV_Log_Ret_btc']]

)
df_btc_hv

# print_dataframe_info(df_btc_hv, "HV - Volatilidade Histórica do BTC")

,Data_UTC,HV_Log_Ret_btc
1,2017-01-01,-0.010314
2,2017-01-02,-0.231446
3,2017-01-03,-0.170868
4,2017-01-04,0.457512
5,2017-01-05,0.582462
...,...,...
3131,2025-07-28,0.013512
3132,2025-07-29,0.000524
3133,2025-07-30,-0.013728
3134,2025-07-31,0.135608


## Funding Rate (Futuros)
- A Funding Rate é o mecanismo que mantém o preço dos contratos futuros perpétuos alinhado ao preço à vista (Spot).
    - Positiva: Os comprados (Longs) pagam os vendidos (Shorts). Indica que a maioria está apostando na alta (otimismo/ganância).
    - Negativa: Os vendidos pagam os comprados. Indica que a maioria está apostando na queda (pessimismo/medo).

- Fonte: https://academy.santiment.net/metrics/funding-rates-aggregated/#definition

### HIPÓTESES
- Na Era Varejo (Período 1), Alta Correlação com Colapsos. O varejo tende a usar alavancagem excessiva irracionalmente. Taxas de Funding muito altas (positivas) devem anteceder quedas bruscas na Dominância ou no preço, pois indicam "topo de euforia".
Se o Feature Importance (SHAP) da Funding Rate for alto aqui, confirma que o mercado era movido por especulação alavancada de pessoas físicas ("gambling").

- Na Era Institucional (Período 2), Efeito Amortecido ou Arbitragem. Instituições usam a Funding Rate para estratégias de Cash and Carry (Delta Neutral). Elas vendem o futuro e compram o spot para receber a taxa.
Se a Funding Rate perder importância preditiva ou mudar de sinal no impacto, sugere que o capital institucional está usando a alavancagem para gerar renda fixa (yield), e não apenas para apostas direcionais.

### TRATAMENTO
- Primeira Diferença (Diff)
- Justificativa: Embora a taxa oscile em torno de zero, a diferença captura a mudança na agressividade da alavancagem (aceleração do otimismo ou do medo). Para modelos VAR, remover a autocorrelação da taxa em nível é essencial.

In [6]:
df_funding_cexs = pd.read_csv(r"raw/2017_fundingRateCEXs.csv")

# Tratamento Inicial
df_funding_tratado = (
    df_funding_cexs
    .assign(Data_UTC = lambda df: pd.to_datetime(df['Date']).dt.strftime("%Y-%m-%d"))
    
    # 1. Média entre Exchanges (Robustez)
    # O parametro numeric_only=True evita erros se houver colunas de texto perdidas
    # O pandas ignora NaNs na média automaticamente (ex: se só tem BitMEX em 2016, usa só BitMEX)
    .assign(total_funding_rate_btc = lambda df: df[[
        'BitMEX Funding Rate', 
        'Binance Funding Rate (USDT)',
        'DyDx Exchange Funding Rates', 
        'Deribit Exchange Funding Rates'
    ]].mean(axis=1, numeric_only=True),
    Funding_Rate_Diff_btc = lambda df: df['total_funding_rate_btc'].diff()
    )

    .query("Data_UTC > '2016-12-31'")
    [['Data_UTC', 'total_funding_rate_btc', 'Funding_Rate_Diff_btc']] # Seleciona apenas o necessário
)

df_funding_tratado

,Data_UTC,total_funding_rate_btc,Funding_Rate_Diff_btc
232,2017-01-01,0.001116,0.001537
233,2017-01-02,0.001892,0.000776
234,2017-01-03,0.001352,-0.000540
235,2017-01-04,0.001414,0.000062
236,2017-01-05,0.003750,0.002336
...,...,...,...
3516,2025-12-29,0.000070,0.000020
3517,2025-12-30,0.000055,-0.000015
3518,2025-12-31,0.000057,0.000002
3519,2026-01-01,0.000036,-0.000021


## Miner Net Position Change (Supply Held) BTC
- Mede a quantidade total de Bitcoin mantida em endereços conhecidos de mineradores e pools de mineração.
Para o modelo, utiliza-se a variação (Net Position Change) para identificar acumulação ou distribuição.

- Mineradores são os únicos "vendedores naturais" e compulsórios do mercado (precisam vender para cobrir custos de eletricidade/Hardware). O aumento das reservas indica acumulação (otimismo), enquanto a queda brusca indica "Capitulation" (medo/necessidade de caixa).
- Fonte: https://newhedge.io/bitcoin/miner-reserves

### HIPÓTESES
- Varejo: Alta Causalidade. No período pré-2020, o mercado tinha menos liquidez. Grandes vendas de mineradores causavam "choques de oferta" significativos, derrubando o preço e a dominância do BTC. Espera-se alto SHAP value.

- Institucional: Menor Relevância. Com a entrada de trilhões de dólares via ETFs e tesourarias, a pressão de venda dos mineradores (aprox. 450 BTC/dia atualmente) é facilmente absorvida pelo volume institucional. Espera-se que perca importância para variáveis Macro (VIX/DXY).

### TRATAMENTO
- Precisamos ver a Variação Líquida da Posição (Net Position Change). Se o saldo de ontem era 2.277k e hoje é 2.279k, eles acumularam +2k BTC (bullish). Se caiu, venderam (bearish). O Log-Retorno é perfeito para isso.

In [7]:
df_supply_btc = pd.read_csv(rf"raw/2009_supply_circulation_btc.csv")

df_supply_held_by = (pd.read_csv(rf"raw/2010_supply_held_miners_whales.csv")
                     
                    .merge(df_supply_btc, how='left', on='Date')
                    .assign(Data_UTC = lambda df: pd.to_datetime(df['Date'], utc=True))
                    .assign(Data_UTC = lambda df: df['Data_UTC'].dt.strftime("%Y-%m-%d"))
                    .rename(columns={'Total Supply': 'Total_Supply_btc',
                                     'Supply held by Miners': 'Supply_Held_by_Miners_btc'})
                                     
                    [['Data_UTC', 'Supply_Held_by_Miners_btc', 'Total_Supply_btc']]
 
)

df_supply_held_by_miners_btc =(
    df_periodo
        .merge(df_supply_held_by, how='left', on='Data_UTC')
        .assign(Miner_Net_Pos_Change = lambda df: np.log(df['Supply_Held_by_Miners_btc']) - np.log(df['Supply_Held_by_Miners_btc'].shift(1)))
        .query("Data_UTC > '2016-12-31'")
        [['Data_UTC','Supply_Held_by_Miners_btc','Miner_Net_Pos_Change']]

)
df_supply_held_by_miners_btc

# print_dataframe_info(df_supply_held_by_miners_btc, "Supply Held by Miners - BTC")

,Data_UTC,Supply_Held_by_Miners_btc,Miner_Net_Pos_Change
1,2017-01-01,2.279762e+06,0.000938
2,2017-01-02,2.281432e+06,0.000732
3,2017-01-03,2.275769e+06,-0.002485
4,2017-01-04,2.313795e+06,0.016571
5,2017-01-05,2.308724e+06,-0.002194
...,...,...,...
3131,2025-07-28,1.700392e+06,-0.000576
3132,2025-07-29,1.701050e+06,0.000387
3133,2025-07-30,1.702182e+06,0.000665
3134,2025-07-31,1.699895e+06,-0.001344


## Supply on Exchanges (as % of total supply)
- É o "termômetro do medo". Quando participantes enviam ativos para corretoras em massa, a intenção primária é vender ou alavancar (short). A métrica captura esses movimentos de manada ("herd behavior") em direção à liquidez das exchanges.

- Proxy de "Demanda por Venda". Não deve ser lida apenas como estoque estático, mas como um indicador de fluxo correlacionado. Picos nessa métrica (podendo atingir correlações de até 20x) sinalizam que múltiplos endereços estão enviando BTC simultaneamente para CEXs/DEXs conhecidas.

- Fonte: https://academy.santiment.net/metrics/supply-on-or-outside-exchanges/

### HIPÓTESES
- Varejo: Correlação Positiva com Volatilidade/Crash. Na era Varejo (pré-2020), picos de envio para exchanges geralmente sinalizavam "Panic Selling" ou rotação para Altcoins (aumentando o risco e diminuindo a dominância do BTC de forma reativa).

- Institucional: Mudança Estrutural (Inversão). A partir de 2020, observou-se uma tendência secular de saída de moedas das exchanges (Choque de Oferta), mesmo durante a alta de preços. A hipótese é que a redução dessa métrica agora causa aumento na dominância e preço (escassez), pois instituições compram e custodiam a longo prazo (ETFs/Tesourarias).

### TRATAMENTO
- Natureza do Dado: É uma porcentagem (Razão) que varia lentamente (tendência secular), mas tem variações diárias que representam o Fluxo Líquido (Net Flow).
- Problema: A série em nível (ex: 15%, 14%) não é estacionária (tem tendência de baixa clara pós-2020).
- **Solução: Usar a Primeira Diferença (Diff).**
- Se o resultado é positivo, houve Inflow (entrada líquida de moedas para venda). Se negativo, houve Outflow (saída para custódia/hold). Isso transforma a variável de "Estoque" para "Fluxo", que é o que impacta o preço no curto prazo.

In [8]:
df_historical_balance = (pd.read_csv(rf"raw/2010_supply_on_exchanges_perc.csv")

                         .assign(Data_UTC = lambda df: pd.to_datetime(df['Date'], utc=True))
                         .assign(Data_UTC = lambda df: df['Data_UTC'].dt.strftime("%Y-%m-%d"))


[['Data_UTC',f'Supply on Exchanges (as % of total supply)']]
 
)

df_supply_held_by_miners_btc =(
    df_periodo
        .merge(df_historical_balance, how='left', on='Data_UTC')
        .rename(columns={f'Supply on Exchanges (as % of total supply)': 'Supply_on_Exchanges_Perc_btc'})
        .assign(Exchange_Supply_Diff_btc = lambda df: df['Supply_on_Exchanges_Perc_btc'].diff())
        .query("Data_UTC > '2016-12-31'")
        [['Data_UTC','Supply_on_Exchanges_Perc_btc','Exchange_Supply_Diff_btc']]

)
df_supply_held_by_miners_btc

# print_dataframe_info(df_historical_balance, "Supply on Exchanges (as % of total supply) - BTC")

,Data_UTC,Supply_on_Exchanges_Perc_btc,Exchange_Supply_Diff_btc
1,2017-01-01,3.402792,0.028501
2,2017-01-02,3.436960,0.034168
3,2017-01-03,3.424486,-0.012473
4,2017-01-04,3.665006,0.240520
5,2017-01-05,3.700899,0.035892
...,...,...,...
3131,2025-07-28,6.155754,-0.013170
3132,2025-07-29,6.154849,-0.000905
3133,2025-07-30,6.156462,0.001612
3134,2025-07-31,6.149945,-0.006516


## SPX-BTC Rolling Correlation (30d)
- Mede o afastamento relativo entre o preço do Bitcoin e o índice S&P 500. Se o valor aumenta, o Bitcoin está se comportando de forma idiossincrática (única). Se diminui ou fica estável, ele está "acoplado" ao mercado de ações.

- Testa diretamente a hipótese de "Acoplamento Institucional". O Bitcoin deixou de ser um ativo não correlacionado para virar um ativo de "Risco Macro"?

- Fonte: https://academy.santiment.net/metrics/btc-and-s-and-p-500-price-divergence

### HIPÓTESES
- Varejo: Baixa Relevância ou Ruído. Antes de 2020, o Bitcoin era um ativo de nicho. Sua divergência do S&P 500 era o padrão (não correlação), portanto, essa métrica não deve explicar grandes mudanças na dominância.

- Institucional: Alta Relevância (Sinal de Alerta). Pós-2020, investidores tratam BTC como "Tech Stock". Momentos de alta divergência indicam eventos cripto-nativos graves (ex: Colapso da FTX ou Luna), o que deve impactar brutalmente a dominância (Fuga para Qualidade ou Fuga do Cripto).

### TRATAMENTO
- Correlação da média móvel do log-retorno do preco da SP500 e o BTC
- Justificativa: Como a correlação varia entre -1 e 1, ela já é estacionária na maioria das janelas. O nível indica o regime (Risk-on/Risk-off). 

In [9]:
df_sp500 = (pd.read_csv(rf"/Users/baia/Desktop/PYTHON/mba_dsa_usp_esalq/TCC/data/dados_macro/2014_SP500_PRICE.csv")

                         .assign(Data_UTC = lambda df: pd.to_datetime(df['time'], utc=True))
                         .assign(Data_UTC = lambda df: df['Data_UTC'].dt.strftime("%Y-%m-%d"))

                         .rename(columns={'close':'sp500_price'})


[['Data_UTC','sp500_price']]
 
)

df_sp500

df_sp500_btc_divergence_btc =(
    df_periodo
        .merge(df_sp500, how='left', on='Data_UTC')
        .merge(df_btc_price, how='left', on='Data_UTC')

        # Pois o preço do SP500 não está diário, preenche os valores faltantes com o último valor conhecido
        .assign(sp500_price = lambda df: df['sp500_price'].ffill())
        .assign(BTC_Log_Ret = lambda df: np.log(df['close']/df['close'].shift(1)))
        .assign(SPX_Log_Ret = lambda df: np.log(df['sp500_price']/df['sp500_price'].shift(1)))

        .assign(BTC_SPX_Corr_30d = lambda df: df['BTC_Log_Ret'].rolling(window=30).corr(df['SPX_Log_Ret']))
        .assign(BTC_SPX_Corr_30d = lambda df: df['BTC_SPX_Corr_30d'].fillna(method='bfill'))     


        .query("Data_UTC > '2016-12-31'")
        [['Data_UTC','BTC_Log_Ret','BTC_SPX_Corr_30d']]

)
df_sp500_btc_divergence_btc
# print_dataframe_info(df_sp500_btc_divergence_btc, "SP500 BTC Divergence - BTC")

/var/folders/h1/_3z4z04x4j17zlfj224w53hr0000gn/T/ipykernel_3073/26225098.py:26: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  .assign(BTC_SPX_Corr_30d = lambda df: df['BTC_SPX_Corr_30d'].fillna(method='bfill'))


,Data_UTC,BTC_Log_Ret,BTC_SPX_Corr_30d
1,2017-01-01,0.026370,-0.162872
2,2017-01-02,0.019043,-0.162872
3,2017-01-03,0.016403,-0.162872
4,2017-01-04,0.090784,-0.162872
5,2017-01-05,-0.116945,-0.162872
...,...,...,...
3131,2025-07-28,-0.011507,0.257894
3132,2025-07-29,-0.001223,0.270594
3133,2025-07-30,-0.001066,0.332494
3134,2025-07-31,-0.017592,0.364173


## Coinbase Premium Index (Manual Ticker)
- diferença percentual entre o preço do Bitcoin na Coinbase Pro (Par BTC/USD) e na Binance (Par BTC/USDT). Indica se o varejo tem mais interesse do que o institucional no bitcoine  vice versa

- A Coinbase é a porta de entrada regulada dos EUA e Institucionais (Tesla, MicroStrategy, Grayscale e ETFs usam a Coinbase). A Binance representa o Varejo Global e especuladores offshore.

- Fonte: https://br.tradingview.com/ (COINBASE:BTCUSD - BINANCE:BTCUSDT)

### HIPÓTESES
- Hipótese da Era Varejo (Pré-2020): Irrelevante ou Ruído. O mercado era dominado pela BitMEX e Binance. O prêmio da Coinbase não ditava a tendência, pois o volume institucional era ínfimo.

- Hipótese da Era Institucional (Pós-2020): Alta Causalidade Positiva. Quando o Coinbase Premium fica positivo (Preço na Coinbase > Binance), significa que instituições americanas estão comprando agressivamente. Isso deve correlacionar fortemente com o aumento da Dominância do BTC (BTC.D), provando a "Fuga para Qualidade" institucional.

### TRATAMENTO
- Ponto de Atenção: Um prêmio de $50 quando o Bitcoin custava $3.000 (2018) é muito mais relevante percentualmente (1.6%) do que um prêmio de $50 quando o Bitcoin custa $60.000 (0.08%).

- No entanto: Para fins de modelagem de curto prazo (variação diária), o sentimento é capturado pela mudança rápida. Se o prêmio salta de $0 para $50 em um dia, houve uma ordem de compra institucional massiva, independente do preço base.

-  Primeira Diferença (Diff)

- Justificativa: Como os valores oscilam em torno de zero (reversão à média), a diferença captura o "choque de demanda" institucional. Um aumento súbito no diferencial sinaliza entrada agressiva de capital regulado.

In [10]:
df_coinbase_premium = (pd.read_csv(rf"raw/2018_coinbase_premium_index.csv")

                         .assign(Data_UTC = lambda df: pd.to_datetime(df['time'], utc=True))
                         .assign(Data_UTC = lambda df: df['Data_UTC'].dt.strftime("%Y-%m-%d"))

                        .rename(columns={'close': 'CB_Premium_USD'}) # Nome mais descritivo que 'close'
                        [['Data_UTC', 'CB_Premium_USD']] # Seleciona apenas o necessário
 
)

df_coinbase_premium_diff =(
    df_periodo
        .merge(df_coinbase_premium, how='left', on='Data_UTC')
        .assign(CB_Premium_Diff_btc = lambda df: df['CB_Premium_USD'].diff())


        .query("Data_UTC > '2019-12-31'")
        [['Data_UTC','CB_Premium_USD','CB_Premium_Diff_btc']]

)
df_coinbase_premium_diff

# print_dataframe_info(df_coinbase_premium, "Coinbase Premium Index - BTC")

,Data_UTC,CB_Premium_USD,CB_Premium_Diff_btc
1096,2020-01-01,-26.52,2.99
1097,2020-01-02,-20.69,5.83
1098,2020-01-03,-10.51,10.18
1099,2020-01-04,-5.48,5.03
1100,2020-01-05,-3.35,2.13
...,...,...,...
3131,2025-07-28,8.27,-41.70
3132,2025-07-29,-17.37,-25.64
3133,2025-07-30,-10.15,7.22
3134,2025-07-31,-2.95,7.20


## MVRV Z-Score (Market Value to Realized Value)
- Mede se o Bitcoin está "caro" ou "barato" em relação ao preço médio que todas as moedas foram compradas (Realized Price). O Z-Score normaliza isso em desvios padrões.

- Ajuda o modelo a entender a psicologia do lucro. Investidores tendem a realizar lucros e mover para Altcoins (derrubando a Dominância) quando o MVRV está muito alto.

- Fonte: https://www.tradingview.com/script/x5cAvOKZ-MVRV-Z-Score/

### HIPÓTESES
- Hipótese da Era Varejo: Gatilho de Altseason. Quando o MVRV Z-Score subia muito (zona de euforia), o varejo vendia BTC para comprar "shitcoins" em busca de retornos maiores. Espera-se correlação negativa com a Dominância (MVRV sobe -> Dominância cai).

- Hipótese da Era Institucional: Amortecimento. Instituições rebalanceiam portfólios com base em métricas de valuation, mas não necessariamente rotacionam para Altcoins arriscadas. O impacto do MVRV alto na queda da dominância deve ser menor ou mais lento nesta era.

### TRATAMENTO
- Primeira Diferença (Diff)
- Justificativa: Como oscilador de reversão à média, a diferença captura a velocidade do encarecimento do ativo. Uma aceleração positiva rápida no MVRV costuma anteceder topos de mercado.

In [12]:
df_mvrv = (pd.read_csv(rf"raw/2017_mvrv_z_score.csv")

                        .assign(Data_UTC = lambda df: pd.to_datetime(df['time'], utc=True))
                        .assign(Data_UTC = lambda df: df['Data_UTC'].dt.strftime("%Y-%m-%d"))
                        .rename(columns={'Plot':'MVRV'})
                        .query("MVRV.isna() == False")

[['Data_UTC','MVRV']]
 
)

df_mvrv_diff =(
    df_periodo
        .merge(df_mvrv, how='left', on='Data_UTC')
        .assign(MVRV_Diff_btc = lambda df: df['MVRV'].diff())


        .query("Data_UTC > '2017-01-02'")
        [['Data_UTC','MVRV','MVRV_Diff_btc']]
)
df_mvrv_diff

# print_dataframe_info(df_mvrv, "MVRV Z-Score - BTC")

,Data_UTC,MVRV,MVRV_Diff_btc
3,2017-01-03,36.302448,0.397958
4,2017-01-04,40.750753,4.448306
5,2017-01-05,34.060526,-6.690227
6,2017-01-06,28.462213,-5.598313
7,2017-01-07,28.761387,0.299174
...,...,...,...
3131,2025-07-28,31.205929,-0.569643
3132,2025-07-29,31.104918,-0.101011
3133,2025-07-30,31.014336,-0.090582
3134,2025-07-31,30.201268,-0.813068


## WHALE TRANSACTION COUNT (> 100K & > 1M)
- Enquanto o varejo faz milhares de pequenas transações, as instituições movem blocos grandes. Esta metrica mostra o numero dtotal de transacoes a qual o valor em USD é maior que o limite estabelecido.

- Fonte: https://academy.santiment.net/metrics/whale-transaction-volume/

### HIPÓTESES
- Era Varejo: Espera-se que esta métrica tenha baixa correlação com a volatilidade ou antecipe apenas grandes dumps (venda de mineradores).

- Era Institucional: Espera-se que picos nesta métrica (acumulação silenciosa) sustentem os preços ("Buy the Dip"), reduzindo a volatilidade, pois instituições compram via OTC ou custódia, diferente do varejo que compra na emoção do order book.

### TRATAMENTO
- A Escala: O max é 11.000 (>1M) e 264.000 (>100k). A média é 1.000 e 44.000. Isso mostra uma distribuição altamente assimétrica (exponencial).
- Implicação para o Modelo: Se usarmos o número absoluto, o modelo vai achar que 2024 é "infinitamente" diferente de 2017 só pelo volume. Necessário nmormalização
- Log-Retorno: Uma contagem que vai de 0 a 200.000 precisa ser comprimida para que o modelo entenda as variações percentuais. Queremos saber se a atividade das baleias aumentou ou diminuiu hoje em relação a ontem.

In [ ]:
df_whale = pd.read_csv(rf"raw/2010_whale_transaction_count_btc.csv")
df_whale['Data_UTC'] = pd.to_datetime(df_whale['Date'], utc=True,).dt.strftime("%Y-%m-%d")
df_whale


df_whale_transaction = (pd.read_csv(rf"raw/2010_whale_transaction_count_btc_2.csv")

                        .assign(Data_UTC = lambda df: pd.to_datetime(df['Date'], utc=True))
                        .assign(Data_UTC = lambda df: df['Data_UTC'].dt.strftime("%Y-%m-%d"))
                        .rename(columns={'Whale Transaction Count (>1m USD)':'Whale_Transaction_Count_1M_btc',
                                         'Whale Transaction Count (>100k USD)':'Whale_Transaction_Count_100k_btc'})

[['Data_UTC','Whale_Transaction_Count_1M_btc','Whale_Transaction_Count_100k_btc']]
 
)

df_whale_transaction_log_ret =(
    df_periodo
        .merge(df_whale_transaction, how='left', on='Data_UTC')

        .fillna(0)
        .assign(Whale_100k_Log_Ret = lambda df: np.log(df['Whale_Transaction_Count_100k_btc'] + 1) - np.log(df['Whale_Transaction_Count_100k_btc'].shift(1) + 1),
                Whale_1M_Log_Ret = lambda df: np.log(df['Whale_Transaction_Count_1M_btc'] + 1) - np.log(df['Whale_Transaction_Count_1M_btc'].shift(1) + 1))


        .query("Data_UTC > '2017-01-02'")
        [['Data_UTC','Whale_100k_Log_Ret','Whale_1M_Log_Ret']]
)
df_whale_transaction_log_ret

,Data_UTC,Whale_100k_Log_Ret,Whale_1M_Log_Ret
3,2017-01-03,0.266011,1.918322
4,2017-01-04,0.283883,-0.367725
5,2017-01-05,0.166293,0.264693
6,2017-01-06,-0.536728,-0.529079
7,2017-01-07,-0.291487,-0.775385
...,...,...,...
3131,2025-07-28,0.378264,0.676219
3132,2025-07-29,0.030633,-0.168348
3133,2025-07-30,0.000334,-0.086928
3134,2025-07-31,0.062892,0.069950
